In [ ]:
# =====================================================================
# 1. DEPENDENCY INSTALLATION & SYSTEM SETUP
# =====================================================================
!pip install torch torchvision diffusers clean-fid torchmetrics transformers datasets gdown -q

import os
import json
import torch
import torch.nn as nn
import numpy as np
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from diffusers import DDPMPipeline, DDIMScheduler
from cleanfid import fid
from transformers import CLIPProcessor, CLIPModel
from datasets import load_dataset
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_DIR = "./dsg_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

# Define sample directories
os.makedirs("samples/unguided",  exist_ok=True)
os.makedirs("samples/dsg",       exist_ok=True)

# =====================================================================
# 2. DISCRIMINATOR GUIDANCE ARCHITECTURE & DATA PREP
# =====================================================================
transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
loader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

class Generator(nn.Module):
    def __init__(self, nz=100, ngf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf*4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),   nn.ReLU(True),
            nn.ConvTranspose2d(ngf, 3,       4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, x): return self.net(x)

class Discriminator(nn.Module):
    def __init__(self, ndf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,      ndf,   4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf,    ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2,  ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4,  ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8,  1,     2, 1, 0, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x): return self.net(x).view(-1, 1)

def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.BatchNorm2d)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)

# Initialize guidance evaluator
G = Generator(100).to(device)
D_cifar = Discriminator().to(device)
G.apply(weights_init)
D_cifar.apply(weights_init)

opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D_cifar.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce = nn.BCELoss()

# QUICK RUN CHANGE: Reduced from 15 to 3 epochs for speed verification
N_EPOCHS = 3
print(f"Training DCGAN for {N_EPOCHS} epochs to get our Guidance Classifier...")
for epoch in range(N_EPOCHS):
    for real_imgs, _ in loader:
        real_imgs = real_imgs.to(device)
        b = real_imgs.size(0)

        # Train Discriminator
        D_cifar.zero_grad()
        loss_real = bce(D_cifar(real_imgs), torch.full((b, 1), 1.0, device=device))
        loss_real.backward()

        fake = G(torch.randn(b, 100, 1, 1, device=device))
        loss_fake = bce(D_cifar(fake.detach()), torch.full((b, 1), 0.0, device=device))
        loss_fake.backward()
        opt_D.step()

        # Train Generator
        G.zero_grad()
        loss_G = bce(D_cifar(fake), torch.full((b, 1), 1.0, device=device))
        loss_G.backward()
        opt_G.step()

torch.save(D_cifar.state_dict(), f'{SAVE_DIR}/dcgan_discriminator_cifar10.pt')
print("Discriminator weights saved.")

# =====================================================================
# 3. PIPELINE PATCH & SAMPLING CORE FUNCTIONS
# =====================================================================
print("\nLoading Diffusion Pipeline...")
pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32").to(device)

# CRITICAL FIX 1: Convert DDPMScheduler to DDIMScheduler to safely support 100 timesteps
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

DELTA = 0.1
BATCH = 64

# QUICK RUN CHANGES: Very small subset sizes to verify the loops instantly
N_SAMPLES = 128
N_ABLATION = 64

def dsg_grad_denoised(D, x_t, scheduler, noise_pred, t):
    alpha_bar = scheduler.alphas_cumprod[t]
    x0_hat = (x_t - (1 - alpha_bar).sqrt() * noise_pred) / alpha_bar.sqrt()
    x0_hat = x0_hat.clamp(-1, 1).detach().requires_grad_(True)

    d_out = D(x0_hat).clamp(DELTA, 1 - DELTA)
    log_odds = (torch.log(d_out) - torch.log(1 - d_out)).sum()
    log_odds.backward()
    return x0_hat.grad.detach()

def dsg_sample_batch(pipe, D, batch_size, gamma=0.1, timestep_cutoff=(0.2, 0.8)):
    unet = pipe.unet
    scheduler = pipe.scheduler
    scheduler.set_timesteps(100)
    timesteps = scheduler.timesteps
    T = len(timesteps)

    x_t = torch.randn(batch_size, 3, 32, 32).to(device)

    for i, t in enumerate(timesteps):
        t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)
        with torch.no_grad():
            noise_pred = unet(x_t, t_batch).sample

        step_out = pipe.scheduler.step(noise_pred, t, x_t)
        x_prev = step_out.prev_sample

        progress = i / T
        if timestep_cutoff[0] < progress < timestep_cutoff[1]:
            grad = dsg_grad_denoised(D, x_t, scheduler, noise_pred, t)
            grad_flat = grad.view(batch_size, -1)
            grad_norm = grad_flat.norm(dim=1, keepdim=True).clamp(min=1e-8)
            grad_n = (grad_flat / grad_norm).view_as(grad)

            # CRITICAL FIX 2: Eliminated the multiplication by the massive total image norm (x_norm)
            x_prev = x_prev + gamma * grad_n

        x_t = x_prev.clamp(-1, 1)

    imgs = ((x_t.cpu() + 1) / 2).clamp(0, 1)
    return [transforms.ToPILImage()(imgs[j]) for j in range(batch_size)]

# =====================================================================
# 4. BASELINE PIPELINE EXECUTION (UNGUIDED VS DSG GUIDED)
# =====================================================================
# Generate baseline unguided images
print(f"\nGenerating {N_SAMPLES} unguided samples...")
n_done = 0
while n_done < N_SAMPLES:
    b = min(BATCH, N_SAMPLES - n_done)
    with torch.no_grad():
        out = pipe(batch_size=b, num_inference_steps=100)
    for j, img in enumerate(out.images):
        img.save(f'samples/unguided/{n_done+j:05d}.png')
    n_done += b

# Generate DSG guided images
print(f"Generating {N_SAMPLES} DSG-guided samples (gamma=0.1)...")
n_done = 0
while n_done < N_SAMPLES:
    b = min(BATCH, N_SAMPLES - n_done)
    imgs = dsg_sample_batch(pipe, D_cifar, b, gamma=0.1)
    for j, img in enumerate(imgs):
        img.save(f'samples/dsg/{n_done+j:05d}.png')
    n_done += b

# CRITICAL FIX 3: Evaluate using clean-fid pre-computed dataset benchmarks
print("\nComputing baseline FID scores against standard reference distribution...")
fid_unguided = fid.compute_fid('samples/unguided/', dataset_name="cifar10", dataset_res=32, dataset_split="train")
fid_dsg = fid.compute_fid('samples/dsg/', dataset_name="cifar10", dataset_res=32, dataset_split="train")

print(f"--> Baseline Unguided FID: {fid_unguided:.2f}")
print(f"--> Baseline DSG Guided FID: {fid_dsg:.2f}")

# =====================================================================
# 5. HYPERPARAMETER ABLATION SWEEPS
# =====================================================================
GAMMAS = [0.0, 0.1, 0.4]  # Shortened list for fast execution check
gamma_results = {}

print(f"\nRunning guidance scale gamma factor ablation sweeps...")
for gamma in GAMMAS:
    out_dir = f'samples/ablation_gamma_{gamma}'
    os.makedirs(out_dir, exist_ok=True)
    n_done = 0
    while n_done < N_ABLATION:
        b = min(BATCH, N_ABLATION - n_done)
        imgs = dsg_sample_batch(pipe, D_cifar, b, gamma=gamma)
        for j, img in enumerate(imgs):
            img.save(f'{out_dir}/{n_done+j:05d}.png')
        n_done += b

    score = fid.compute_fid(out_dir, dataset_name="cifar10", dataset_res=32, dataset_split="train")
    gamma_results[gamma] = score
    print(f"  Gamma: {gamma:.2f} → FID: {score:.2f}")

with open('results_ablation_gamma.json', 'w') as f:
    json.dump(gamma_results, f)

best_gamma = min(gamma_results, key=gamma_results.get)

# Timestep Cutoffs Ablation Range Sweeps
CUTOFFS = [(0.1, 0.4), (0.2, 0.8)]  # Shortened list for fast execution check
cutoff_results = {}

print(f"\nRunning timestep filter ablation sweeps (Using Best Gamma={best_gamma})...")
for low, high in CUTOFFS:
    label = f"{low}-{high}"
    out_dir = f'samples/ablation_cutoff_{label}'
    os.makedirs(out_dir, exist_ok=True)
    n_done = 0
    while n_done < N_ABLATION:
        b = min(BATCH, N_ABLATION - n_done)
        imgs = dsg_sample_batch(pipe, D_cifar, b, gamma=best_gamma, timestep_cutoff=(low, high))
        for j, img in enumerate(imgs):
            img.save(f'{out_dir}/{n_done+j:05d}.png')
        n_done += b

    score = fid.compute_fid(out_dir, dataset_name="cifar10", dataset_res=32, dataset_split="train")
    cutoff_results[label] = score
    print(f"  Cutoff Window: {label} → FID: {score:.2f}")

with open('results_ablation_cutoff.json', 'w') as f:
    json.dump(cutoff_results, f)

best_cutoff = min(cutoff_results, key=cutoff_results.get)

# =====================================================================
# 6. HIGHER RESOLUTION SELECTION (CELEBA EVALUATION WITH CLIP)
# =====================================================================
print("\nLoading CelebA Evaluation pipelines and vision alignment metrics...")
celeba_pipe = DDPMPipeline.from_pretrained("google/ddpm-celebahq-256").to(device)
celeba_pipe.scheduler = DDIMScheduler.from_config(celeba_pipe.scheduler.config)

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def clip_score(images, prompt):
    inputs = clip_processor(text=[prompt], images=images, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = clip_model(**inputs)
    logits_per_image = outputs.logits_per_image
    return logits_per_image.mean().item()

def dsg_grad_celeba(D, x_t, scheduler, noise_pred, t):
    alpha_bar = scheduler.alphas_cumprod[t]
    x0_hat = (x_t - (1 - alpha_bar).sqrt() * noise_pred) / alpha_bar.sqrt()
    x0_hat = x0_hat.clamp(-1, 1).detach().requires_grad_(True)

    x0_resized = nn.functional.interpolate(x0_hat, size=(32, 32), mode='bilinear', align_corners=False)

    d_out = D(x0_resized).clamp(DELTA, 1 - DELTA)
    log_odds = (torch.log(d_out) - torch.log(1 - d_out)).sum()
    log_odds.backward()
    return x0_hat.grad.detach()

def celeba_sample_batch(pipe, D, batch_size, gamma=0.1, timestep_cutoff=(0.2, 0.8)):
    unet = pipe.unet
    scheduler = pipe.scheduler
    scheduler.set_timesteps(20)  # QUICK RUN CHANGE: 20 steps for fast pipeline verification
    timesteps = scheduler.timesteps
    T = len(timesteps)

    x_t = torch.randn(batch_size, 3, 256, 256).to(device)

    for i, t in enumerate(timesteps):
        t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)
        with torch.no_grad():
            noise_pred = unet(x_t, t_batch).sample

        step_out = pipe.scheduler.step(noise_pred, t, x_t)
        x_prev = step_out.prev_sample

        progress = i / T
        if timestep_cutoff[0] < progress < timestep_cutoff[1]:
            grad = dsg_grad_celeba(D, x_t, scheduler, noise_pred, t)
            grad_flat = grad.view(batch_size, -1)
            grad_norm = grad_flat.norm(dim=1, keepdim=True).clamp(min=1e-8)
            grad_n = (grad_flat / grad_norm).view_as(grad)
            x_prev = x_prev + gamma * grad_n

        x_t = x_prev.clamp(-1, 1)

    imgs = ((x_t.cpu() + 1) / 2).clamp(0, 1)
    return [transforms.ToPILImage()(imgs[j]) for j in range(batch_size)]

print("Running CelebA Prompt Similarity tests...")
with torch.no_grad():
    unguided_celeba = celeba_pipe(batch_size=4, num_inference_steps=20).images
guided_celeba = celeba_sample_batch(celeba_pipe, D_cifar, batch_size=4, gamma=0.1)

prompt = "a photograph of a smiling person"
clip_unguided = clip_score(unguided_celeba, prompt)
clip_guided = clip_score(guided_celeba, prompt)

# =====================================================================
# 7. FINAL LOG METRIC SUMMARY REPORTING
# =====================================================================
print("\n" + "="*60)
print("QUICK-RUN VERIFICATION SUMMARY COMPLETED")
print("="*60)
print(f"Table 1 — Quick CIFAR-10 FID Check")
print(f"  Unguided Baseline : FID = {fid_unguided:.2f}")
print(f"  DSG Guided Group  : FID = {fid_dsg:.2f}")

print("\nTable 2 — Guidance Sweep (Gamma)")
for g, f_score in sorted(gamma_results.items()):
    marker = " ← OPTIMAL" if g == best_gamma else ""
    print(f"  gamma={g:.2f} : FID={f_score:.2f}{marker}")

print("\nTable 3 — Timestep Window Filter")
for c, f_score in cutoff_results.items():
    marker = " ← OPTIMAL" if c == best_cutoff else ""
    print(f"  cutoff window={c} : FID={f_score:.2f}{marker}")

print("\nTable 4 — Text Alignment Prompt Profile (CLIP)")
print(f"  Baseline Unguided Score : {clip_unguided:.4f}")
print(f"  Conditional DSG Score   : {clip_guided:.4f}")
print("="*60)

Training DCGAN for 3 epochs to get our Guidance Classifier...
Discriminator weights saved.

Loading Diffusion Pipeline...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Running CelebA Prompt Similarity tests...


  0%|          | 0/20 [00:00<?, ?it/s]


QUICK-RUN VERIFICATION SUMMARY COMPLETED
Table 1 — Quick CIFAR-10 FID Check
  Unguided Baseline : FID = 138.71
  DSG Guided Group  : FID = 254.40

Table 2 — Guidance Sweep (Gamma)
  gamma=0.00 : FID=285.84
  gamma=0.10 : FID=270.69 ← OPTIMAL
  gamma=0.40 : FID=278.83

Table 3 — Timestep Window Filter
  cutoff window=0.1-0.4 : FID=273.52 ← OPTIMAL
  cutoff window=0.2-0.8 : FID=287.78

Table 4 — Text Alignment Prompt Profile (CLIP)
  Baseline Unguided Score : 25.3385
  Conditional DSG Score   : 22.5452


In [ ]:
import torch
import torch.nn as nn
import torchvision
import os
import gdown
from diffusers import DDPMPipeline, DDIMScheduler
from cleanfid import fid

import shutil

# WIPE OUT THE POLLUTED DIRECTORIES TO START FRESH
if os.path.exists("samples/unguided_ddim"):
    shutil.rmtree("samples/unguided_ddim")
if os.path.exists("samples/dsg_optimized"):
    shutil.rmtree("samples/dsg_optimized")

# RECREATING EMPTY FOLDERS
os.makedirs("samples/unguided_ddim", exist_ok=True)
os.makedirs("samples/dsg_optimized", exist_ok=True)

# ... [Rest of your generation and clean-fid code here] ...

device = "cuda" if torch.cuda.is_available() else "cpu"
DELTA = 0.1
N_SAMPLES = 1000  # Standard sample size for stable FID evaluation
BATCH = 64

# Define potential paths for your weights
LOCAL_WEIGHTS = 'dcgan_discriminator_cifar10.pt'
DRIVE_WEIGHTS = '/content/drive/MyDrive/dsg_checkpoints/dcgan_discriminator_cifar10.pt'
FILE_ID = '1ajwna_dAOaGt9DmZ1pazgkToiBFWw1Zc'

# Create necessary directories
os.makedirs("samples/unguided_ddim", exist_ok=True)
os.makedirs("samples/dsg_optimized", exist_ok=True)
os.makedirs("data/cifar10_real", exist_ok=True)


# ── 1. LOADING DISCRIMINATOR WITH AUTOMATIC FALLBACKS ─────────────────
D = Discriminator().to(device)

if os.path.exists(LOCAL_WEIGHTS):
    print(f"-> Loading weights from local file: {LOCAL_WEIGHTS}")
    D.load_state_dict(torch.load(LOCAL_WEIGHTS, map_location=device))
elif os.path.exists(DRIVE_WEIGHTS):
    print(f"-> Loading weights from Google Drive: {DRIVE_WEIGHTS}")
    D.load_state_dict(torch.load(DRIVE_WEIGHTS, map_location=device))
else:
    print("-> Weights not found locally or in Drive. Downloading via gdown...")
    url = f'https://drive.google.com/uc?id={FILE_ID}'
    gdown.download(url, LOCAL_WEIGHTS, quiet=False)
    D.load_state_dict(torch.load(LOCAL_WEIGHTS, map_location=device))

D.eval()
print("Discriminator successfully loaded and set to evaluation mode.\n")


# ── 2. LOAD PIPELINE & UPGRADE TO DDIM SCHEDULER ──────────────────────
print("Loading diffusion pipeline components...")
pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32").to(device)

# SWAP TO DDIM: Essential to get high baseline quality at 100 inference steps
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)


# ── 3. SAVE REAL CIFAR-10 REFERENCE IMAGES FOR FID ────────────────────
print("\nChecking real CIFAR-10 reference dataset...")
num_existing_real = len(os.listdir('data/cifar10_real'))
if num_existing_real < N_SAMPLES:
    print(f"Saving {N_SAMPLES} real images to 'data/cifar10_real' for FID evaluation...")
    transform = torchvision.transforms.Compose([torchvision.transforms.ToTensor()])
    real_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    for i, (img, _) in enumerate(real_set):
        if i >= N_SAMPLES: break
        torchvision.utils.save_image(img, f'data/cifar10_real/{i:05d}.png')
    print("Real dataset prepared.")
else:
    print(f"Found {num_existing_real} reference images. Skipping extraction.")


# ── 4. OPTIMIZED DSG GUIDANCE TRAJECTORY ─────────────────────────────
def dsg_grad_denoised(D, x_t, scheduler, noise_pred, t):
    alpha_bar = scheduler.alphas_cumprod[t]
    # Predict the fully denoised clean image x0
    x0_hat = (x_t - (1 - alpha_bar).sqrt() * noise_pred) / alpha_bar.sqrt()
    x0_hat = x0_hat.clamp(-1, 1).detach().requires_grad_(True)

    # Compute score gradients through log-odds
    d_out = D(x0_hat).clamp(DELTA, 1 - DELTA)
    log_odds = (torch.log(d_out) - torch.log(1 - d_out)).sum()
    log_odds.backward()
    return x0_hat.grad.detach()

def dsg_sample_batch_optimized(pipe, D, batch_size, gamma=0.02, timestep_cutoff=(0.5, 0.9)):
    unet = pipe.unet
    scheduler = pipe.scheduler
    scheduler.set_timesteps(100) # Fast trajectory
    timesteps = scheduler.timesteps
    T = len(timesteps)

    x_t = torch.randn(batch_size, 3, 32, 32).to(device)

    for i, t in enumerate(timesteps):
        t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)
        with torch.no_grad():
            noise_pred = unet(x_t, t_batch).sample

        # Standard reverse step
        step_out = pipe.scheduler.step(noise_pred, t, x_t)
        x_prev = step_out.prev_sample

        # Restrict guidance to late structural window where x0_hat is reliable
        progress = i / T
        if timestep_cutoff[0] < progress < timestep_cutoff[1]:
            grad = dsg_grad_denoised(D, x_t, scheduler, noise_pred, t)

            # Unit norm stabilization per batch element
            grad_flat = grad.view(batch_size, -1)
            grad_norm = grad_flat.norm(dim=1, keepdim=True).clamp(min=1e-8)
            grad_n = (grad_flat / grad_norm).view_as(grad)

            # Dynamic scaling proportional to the current latent magnitude
            x_norm = x_t.view(batch_size, -1).norm(dim=1).view(batch_size, 1, 1, 1)
            x_prev = x_prev + gamma * x_norm * grad_n

        x_t = x_prev.clamp(-1, 1)

    imgs = ((x_t.cpu() + 1) / 2).clamp(0, 1)
    return [torchvision.transforms.ToPILImage()(imgs[j]) for j in range(batch_size)]


# ── 5. GENERATE IMAGE SAMPLES ─────────────────────────────────────────
print(f"\nGenerating {N_SAMPLES} Unguided DDIM baseline samples...")
n_done = 0
while n_done < N_SAMPLES:
    b = min(BATCH, N_SAMPLES - n_done)
    with torch.no_grad():
        out = pipe(batch_size=b, num_inference_steps=100)
    for j, img in enumerate(out.images):
        img.save(f'samples/unguided_ddim/{n_done+j:05d}.png')
    n_done += b
print("Unguided generation complete.")

print(f"\nGenerating {N_SAMPLES} Optimized DSG-guided samples...")
n_done = 0
while n_done < N_SAMPLES:
    b = min(BATCH, N_SAMPLES - n_done)
    imgs = dsg_sample_batch_optimized(pipe, D, b, gamma=0.02, timestep_cutoff=(0.5, 0.9))
    for j, img in enumerate(imgs):
        img.save(f'samples/dsg_optimized/{n_done+j:05d}.png')
    n_done += b
print("DSG-Guided generation complete.")


# ── 6. COMPUTE FID SCORES ────────────────────────────────────────────
print("\nComputing absolute FID scores using clean-fid...")
fid_unguided = fid.compute_fid('samples/unguided_ddim/', 'data/cifar10_real/')
fid_dsg_opt = fid.compute_fid('samples/dsg_optimized/', 'data/cifar10_real/')

print("\n" + "="*55)
print("               FINAL QUALITY VERIFICATION               ")
print("="*55)
print(f"  Baseline Unguided DDIM (100 steps) : FID = {fid_unguided:.2f}")
print(f"  Optimized DSG-Guided   (100 steps) : FID = {fid_dsg_opt:.2f}")
print("-"*55)
print(f"  Net FID Improvement                : {fid_unguided - fid_dsg_opt:.2f} points")
print("="*55)

-> Loading weights from local file: dcgan_discriminator_cifar10.pt
Discriminator successfully loaded and set to evaluation mode.

Loading diffusion pipeline components...


Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.



Checking real CIFAR-10 reference dataset...
Found 10000 reference images. Skipping extraction.

Generating 1000 Unguided DDIM baseline samples...


  0%|          | 0/100 [00:00<?, ?it/s]

...

compute FID between two folders
Found 1000 images in the folder samples/dsg_optimized/


FID  : 100%|██████████| 32/32 [00:12<00:00,  2.53it/s]


Found 10000 images in the folder data/cifar10_real/


FID  : 100%|██████████| 313/313 [00:57<00:00,  5.46it/s]



               FINAL QUALITY VERIFICATION               
  Baseline Unguided DDIM (100 steps) : FID = 42.87
  Optimized DSG-Guided   (100 steps) : FID = 191.26
-------------------------------------------------------
  Net FID Improvement                : -148.39 points


In [ ]:
import os
import shutil
import torch
import torch.nn as nn
import numpy as np
import torchvision
from diffusers import DDPMPipeline
from cleanfid import fid

# ==========================================
# 1. HARDWARE & DIRECTORY CLEANUP
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running execution pipeline on: {device.upper()}")

# Clean old directories to prevent FID evaluation caching pollution
dirs_to_clean = ["samples/unguided_ddim", "samples/dsg_optimized"]
for folder in dirs_to_clean:
    if os.path.exists(folder):
        print(f"Clearing old cache: {folder}")
        shutil.rmtree(folder)
    os.makedirs(f"{folder}/FID", exist_ok=True)

os.makedirs("data/cifar10_real/FID", exist_ok=True)

# ==========================================
# 2. RUNTIME CONFIGURATION
# ==========================================
N_SAMPLES = 1000       # 1000 for verification, change to 10000 for publication
BATCH = 64
DELTA = 0.1
GAMMA_DSG = 0.2        # Scale factor when modifying the noise prediction space
CUTOFF = (0.3, 0.8)    # Guide during the crucial structural phase (30% to 80% progress)
LOCAL_WEIGHTS = 'dcgan_discriminator_cifar10.pt'

# ==========================================
# 3. LOAD CORE COMPONENTS
# ==========================================
print("\n[1/5] Loading diffusion pipeline components...")
pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32").to(device)

print(f"[2/5] Loading weights from local file: {LOCAL_WEIGHTS}")
# Reconstruct your Discriminator architecture matching your saved checkpoint
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, kernel_size=2, stride=1, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

D = Discriminator().to(device)
if os.path.exists(LOCAL_WEIGHTS):
    D.load_state_dict(torch.load(LOCAL_WEIGHTS, map_location=device))
    D.eval()
    print("Discriminator successfully loaded and set to evaluation mode.")
else:
    raise FileNotFoundError(f"Missing {LOCAL_WEIGHTS}! Please check your file paths.")

# ==========================================
# 4. REFERENCE DATASET EXTRACTION
# ==========================================
print("\n[3/5] Checking real CIFAR-10 reference dataset...")
if len(os.listdir("data/cifar10_real/FID")) < N_SAMPLES:
    transform = torchvision.transforms.Compose([torchvision.transforms.ToTensor()])
    real_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    for i, (img, _) in enumerate(real_set):
        if i >= N_SAMPLES: break
        torchvision.utils.save_image(img, f'data/cifar10_real/FID/{i:05d}.png')
    print(f"Extracted {N_SAMPLES} reference images.")
else:
    print(f"Found {len(os.listdir('data/cifar10_real/FID'))} reference images. Skipping extraction.")

# ==========================================
# 5. MATHEMATICALLY SOUND DSG LOGIC
# ==========================================
def dsg_grad_denoised(D, x_t, scheduler, noise_pred, t):
    """
    Predicts x0_hat analytically from x_t and noise_pred,
    then backpropagates the discriminator's log-odds score.
    """
    alpha_bar = scheduler.alphas_cumprod[t]

    # Mathematical projection down to the clean manifold estimate
    x0_hat = (x_t - (1 - alpha_bar).sqrt() * noise_pred) / alpha_bar.sqrt()
    x0_hat = x0_hat.clamp(-1, 1).detach().requires_grad_(True)

    # Compute discriminator score in log-odds space to maintain stable gradient signals
    d_out = D(x0_hat).clamp(DELTA, 1 - DELTA)
    log_odds = (torch.log(d_out) - torch.log(1 - d_out)).sum()
    log_odds.backward()

    return x0_hat.grad.detach()

def dsg_sample_batch_optimized(pipe, D, batch_size, gamma=0.2, timestep_cutoff=(0.3, 0.8)):
    """
    Generates guided samples by safely altering the score-space
    (noise prediction) instead of causing latent manifold displacement.
    """
    unet = pipe.unet
    scheduler = pipe.scheduler
    scheduler.set_timesteps(100) # Fast evaluation timestep configuration
    timesteps = scheduler.timesteps
    T = len(timesteps)

    # Standard Gaussian Latent Initialization
    x_t = torch.randn(batch_size, 3, 32, 32).to(device)

    for i, t in enumerate(timesteps):
        t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)

        # 1. Unconditional score prediction from UNet
        with torch.no_grad():
            noise_pred = unet(x_t, t_batch).sample

        progress = i / T
        # 2. Score correction window (Plug-and-play guidance)
        if timestep_cutoff[0] < progress < timestep_cutoff[1]:
            # Get raw discriminator score gradients w.r.t clean projection
            grad = dsg_grad_denoised(D, x_t, scheduler, noise_pred, t)

            # THE THEORETICAL CORRECTION: Shift the noise prediction vector.
            # This directly aligns with score-matching theory, where guidance updates
            # are naturally scaled by the noise schedule multiplier (sqrt(1 - alpha_bar)).
            alpha_bar = scheduler.alphas_cumprod[t]
            noise_pred = noise_pred - (1 - alpha_bar).sqrt() * gamma * grad

        # 3. Transition to previous step via the scheduler using the corrected score
        with torch.no_grad():
            step_out = pipe.scheduler.step(noise_pred, t, x_t)
            x_t = step_out.prev_sample.clamp(-1, 1)

    # Transform tensor lattice to standardized PIL Images
    imgs = ((x_t.cpu() + 1) / 2).clamp(0, 1)
    return [torchvision.transforms.ToPILImage()(imgs[j]) for j in range(batch_size)]

# ==========================================
# 6. SAMPLING GENERATION LOOPS
# ==========================================
print(f"\n[4/5] Generating {N_SAMPLES} Unguided DDIM baseline samples...")
n_done = 0
while n_done < N_SAMPLES:
    b = min(BATCH, N_SAMPLES - n_done)
    with torch.no_grad():
        # Force the scheduler inside the standard pipe call to 100 steps for fairness
        pipe.scheduler.set_timesteps(100)
        out = pipe(batch_size=b, num_inference_steps=100)
    for j, img in enumerate(out.images):
        img.save(f'samples/unguided_ddim/FID/{n_done+j:05d}.png')
    n_done += b
print("Unguided generation complete.")

print(f"\n[5/5] Generating {N_SAMPLES} Optimized DSG-guided samples...")
n_done = 0
while n_done < N_SAMPLES:
    b = min(BATCH, N_SAMPLES - n_done)
    imgs = dsg_sample_batch_optimized(pipe, D, b, gamma=GAMMA_DSG, timestep_cutoff=CUTOFF)
    for j, img in enumerate(imgs):
        img.save(f'samples/dsg_optimized/FID/{n_done+j:05d}.png')
    n_done += b
print("DSG-Guided generation complete.")

# ==========================================
# 7. FID SCORE EVALUATION
# ==========================================
print("\nComputing absolute FID scores using clean-fid...")
fid_unguided = fid.compute_fid('samples/unguided_ddim/FID', 'data/cifar10_real/FID')
fid_dsg = fid.compute_fid('samples/dsg_optimized/FID', 'data/cifar10_real/FID')

print("\n" + "="*55)
print("               FINAL QUALITY VERIFICATION               ")
print("="*55)
print(f"  Baseline Unguided DDIM (100 steps) : FID = {fid_unguided:.2f}")
print(f"  Optimized DSG-Guided   (100 steps) : FID = {fid_dsg:.2f}")
print("-"*55)
print(f"  Net FID Improvement                : {fid_unguided - fid_dsg:+.2f} points")
print("="*55)

Running execution pipeline on: CUDA
Clearing old cache: samples/unguided_ddim
Clearing old cache: samples/dsg_optimized

[1/5] Loading diffusion pipeline components...


Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


[2/5] Loading weights from local file: dcgan_discriminator_cifar10.pt
Discriminator successfully loaded and set to evaluation mode.

[3/5] Checking real CIFAR-10 reference dataset...
Extracted 1000 reference images.

[4/5] Generating 1000 Unguided DDIM baseline samples...


  0%|          | 0/100 [00:00<?, ?it/s]

...

compute FID between two folders
Found 1000 images in the folder samples/dsg_optimized/FID


FID FID : 100%|██████████| 32/32 [00:12<00:00,  2.56it/s]


Found 1000 images in the folder data/cifar10_real/FID


FID FID : 100%|██████████| 32/32 [00:11<00:00,  2.70it/s]



               FINAL QUALITY VERIFICATION               
  Baseline Unguided DDIM (100 steps) : FID = 125.09
  Optimized DSG-Guided   (100 steps) : FID = 334.47
-------------------------------------------------------
  Net FID Improvement                : -209.38 points


In [ ]:
import os
import shutil
import torch
import torch.nn as nn
import numpy as np
import torchvision
from diffusers import DDPMPipeline
from cleanfid import fid

# =====================================================================
# 1. HARDWARE INITIALIZATION & CACHE PURGING
# =====================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running execution pipeline on: {device.upper()}")

# Completely wipe clean old sample paths to avoid clean-fid cache pollution
dirs_to_clean = ["samples/unguided_ddim", "samples/dsg_optimized"]
for folder in dirs_to_clean:
    if os.path.exists(folder):
        print(f"Clearing old cache: {folder}")
        shutil.rmtree(folder)
    os.makedirs(f"{folder}/FID", exist_ok=True)

os.makedirs("data/cifar10_real/FID", exist_ok=True)

# =====================================================================
# 2. HYPERPARAMETER TUNING CONFIGURATION
# =====================================================================
N_SAMPLES = 1000       # Run 1,000 for rapid testing; 10,000 for final publication
BATCH = 64
DELTA = 0.1

# THE HYPERPARAMETERS THAT CONTROL MANIFOLD STABILITY:
GAMMA_DSG = 0.1        # Step scale. Keeping this <= 0.2 ensures stable guidance
CUTOFF = (0.5, 0.9)    # Shifted to later timesteps (50%-90% progress) where x0_hat estimates are stable
LOCAL_WEIGHTS = 'dcgan_discriminator_cifar10.pt'

# =====================================================================
# 3. COMPONENT INGESTION (DIFFUSION BACKBONE & DISCRIMINATOR)
# =====================================================================
print("\n[1/5] Loading diffusion pipeline components...")
pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32").to(device)

print(f"[2/5] Loading weights from local file: {LOCAL_WEIGHTS}")
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, kernel_size=2, stride=1, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

D = Discriminator().to(device)
if os.path.exists(LOCAL_WEIGHTS):
    D.load_state_dict(torch.load(LOCAL_WEIGHTS, map_location=device))
    D.eval()
    print("Discriminator successfully loaded and set to evaluation mode.")
else:
    raise FileNotFoundError(f"Missing {LOCAL_WEIGHTS}! Please check your execution directory.")

# =====================================================================
# 4. REFERENCE DATASET VALIDATION
# =====================================================================
print("\n[3/5] Checking real CIFAR-10 reference dataset...")
if len(os.listdir("data/cifar10_real/FID")) < N_SAMPLES:
    transform = torchvision.transforms.Compose([torchvision.transforms.ToTensor()])
    real_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    for i, (img, _) in enumerate(real_set):
        if i >= N_SAMPLES: break
        torchvision.utils.save_image(img, f'data/cifar10_real/FID/{i:05d}.png')
    print(f"Extracted {N_SAMPLES} reference images.")
else:
    print(f"Found {len(os.listdir('data/cifar10_real/FID'))} reference images. Skipping extraction.")

# =====================================================================
# 5. FIXED DSG LOGIC (UNIT VECTOR SCALE IN NOISE-PRED SPACE)
# =====================================================================
def dsg_grad_denoised(D, x_t, scheduler, noise_pred, t):
    """
    Computes log-odds gradients from a clean x0 estimate.
    """
    alpha_bar = scheduler.alphas_cumprod[t]

    # Derivation of clean image manifold estimate
    x0_hat = (x_t - (1 - alpha_bar).sqrt() * noise_pred) / alpha_bar.sqrt()
    x0_hat = x0_hat.clamp(-1, 1).detach().requires_grad_(True)

    # Backprop log-odds score (more mathematically stable than raw probabilities)
    d_out = D(x0_hat).clamp(DELTA, 1 - DELTA)
    log_odds = (torch.log(d_out) - torch.log(1 - d_out)).sum()
    log_odds.backward()

    return x0_hat.grad.detach()

def dsg_sample_batch_optimized(pipe, D, batch_size, gamma=0.1, timestep_cutoff=(0.5, 0.9)):
    """
    Runs the generative loop by treating guidance as a bounded noise-prediction adjustment.
    """
    unet = pipe.unet
    scheduler = pipe.scheduler
    scheduler.set_timesteps(100) # Target baseline 100-step loop
    timesteps = scheduler.timesteps
    T = len(timesteps)

    x_t = torch.randn(batch_size, 3, 32, 32).to(device)

    for i, t in enumerate(timesteps):
        t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)

        with torch.no_grad():
            noise_pred = unet(x_t, t_batch).sample

        progress = i / T
        # Only guide when structural noise settles to minimize early mathematical variance
        if timestep_cutoff[0] < progress < timestep_cutoff[1]:
            grad = dsg_grad_denoised(D, x_t, scheduler, noise_pred, t)

            # --- THE CRITICAL MANIFOLD STABILIZATION FIX ---
            # Reshape, isolate raw magnitudes, and scale as an exact directional unit vector.
            grad_flat = grad.view(batch_size, -1)
            grad_norm = grad_flat.norm(dim=1, keepdim=True).clamp(min=1e-8)
            grad_unit = (grad_flat / grad_norm).view_as(grad)

            # Blend into the noise-prediction dimension cleanly scaled by schedule constants
            alpha_bar = scheduler.alphas_cumprod[t]
            noise_pred = noise_pred - (1 - alpha_bar).sqrt() * gamma * grad_unit

        # Advance latent variable safely using the modified score function
        with torch.no_grad():
            step_out = pipe.scheduler.step(noise_pred, t, x_t)
            x_t = step_out.prev_sample.clamp(-1, 1)

    # Post-process target lattice tensor to PIL format
    imgs = ((x_t.cpu() + 1) / 2).clamp(0, 1)
    return [torchvision.transforms.ToPILImage()(imgs[j]) for j in range(batch_size)]

# =====================================================================
# 6. PIPELINE PROCESSING LOOPS
# =====================================================================
print(f"\n[4/5] Generating {N_SAMPLES} Unguided DDIM baseline samples...")
n_done = 0
while n_done < N_SAMPLES:
    b = min(BATCH, N_SAMPLES - n_done)
    with torch.no_grad():
        pipe.scheduler.set_timesteps(100)
        out = pipe(batch_size=b, num_inference_steps=100)
    for j, img in enumerate(out.images):
        img.save(f'samples/unguided_ddim/FID/{n_done+j:05d}.png')
    n_done += b
print("Unguided generation complete.")

print(f"\n[5/5] Generating {N_SAMPLES} Optimized DSG-guided samples...")
n_done = 0
while n_done < N_SAMPLES:
    b = min(BATCH, N_SAMPLES - n_done)
    imgs = dsg_sample_batch_optimized(pipe, D, b, gamma=GAMMA_DSG, timestep_cutoff=CUTOFF)
    for j, img in enumerate(imgs):
        img.save(f'samples/dsg_optimized/FID/{n_done+j:05d}.png')
    n_done += b
print("DSG-Guided generation complete.")

# =====================================================================
# 7. METRIC EVALUATION (CLEAN-FID)
# =====================================================================
print("\nComputing absolute FID scores using clean-fid...")
fid_unguided = fid.compute_fid('samples/unguided_ddim/FID', 'data/cifar10_real/FID')
fid_dsg = fid.compute_fid('samples/dsg_optimized/FID', 'data/cifar10_real/FID')

print("\n" + "="*55)
print("               FINAL QUALITY VERIFICATION               ")
print("="*55)
print(f"  Baseline Unguided DDIM (100 steps) : FID = {fid_unguided:.2f}")
print(f"  Optimized DSG-Guided   (100 steps) : FID = {fid_dsg:.2f}")
print("-"*55)
print(f"  Net FID Improvement                : {fid_unguided - fid_dsg:+.2f} points")
print("="*55)

Running execution pipeline on: CUDA
Clearing old cache: samples/unguided_ddim
Clearing old cache: samples/dsg_optimized

[1/5] Loading diffusion pipeline components...


Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


[2/5] Loading weights from local file: dcgan_discriminator_cifar10.pt
Discriminator successfully loaded and set to evaluation mode.

[3/5] Checking real CIFAR-10 reference dataset...
Found 1000 reference images. Skipping extraction.

[4/5] Generating 1000 Unguided DDIM baseline samples...


  0%|          | 0/100 [00:00<?, ?it/s]

...

compute FID between two folders
Found 1000 images in the folder samples/dsg_optimized/FID


FID FID : 100%|██████████| 32/32 [00:13<00:00,  2.45it/s]


Found 1000 images in the folder data/cifar10_real/FID


FID FID : 100%|██████████| 32/32 [00:12<00:00,  2.65it/s]



               FINAL QUALITY VERIFICATION               
  Baseline Unguided DDIM (100 steps) : FID = 122.71
  Optimized DSG-Guided   (100 steps) : FID = 329.51
-------------------------------------------------------
  Net FID Improvement                : -206.80 points


In [ ]:
import os
import shutil
import torch
import torch.nn as nn
import numpy as np
import torchvision
from diffusers import DDPMPipeline, DDIMScheduler
from cleanfid import fid
from tqdm import tqdm

# =====================================================================
# 1. HARDWARE INITIALIZATION & CACHE PURGING
# =====================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running execution pipeline on: {device.upper()}")

# Clear old evaluation folders to prevent clean-fid cache pollution
dirs_to_clean = ["samples/unguided_ddim", "samples/dsg_optimized"]
for folder in dirs_to_clean:
    if os.path.exists(folder):
        print(f"Clearing old cache: {folder}")
        shutil.rmtree(folder)
    os.makedirs(f"{folder}/FID", exist_ok=True)

os.makedirs("data/cifar10_real/FID", exist_ok=True)

# =====================================================================
# 2. HYPERPARAMETER CONFIGURATION
# =====================================================================
N_SAMPLES = 1000
BATCH = 64
DELTA = 0.1
GAMMA_DSG = 0.05       # Stable step size for DDIM space guidance
CUTOFF = (0.2, 0.8)    # Active guidance window (20% to 80% of the denoising process)
LOCAL_WEIGHTS = 'dcgan_discriminator_cifar10.pt'

# =====================================================================
# 3. COMPONENT LOADING & SCHEDULER SWAP
# =====================================================================
print("\n[1/5] Loading diffusion pipeline components...")
pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32").to(device)

# CRITICAL FIX: Explicitly swap to DDIMScheduler to allow valid 100-step custom loops
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.scheduler.set_timesteps(100)

print(f"[2/5] Loading weights from local file: {LOCAL_WEIGHTS}")
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, kernel_size=2, stride=1, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

D = Discriminator().to(device)
if os.path.exists(LOCAL_WEIGHTS):
    D.load_state_dict(torch.load(LOCAL_WEIGHTS, map_location=device))
    D.eval()
    print("Discriminator successfully loaded and set to evaluation mode.")
else:
    raise FileNotFoundError(f"Missing {LOCAL_WEIGHTS}!")

# =====================================================================
# 4. REFERENCE DATASET VALIDATION
# =====================================================================
print("\n[3/5] Checking real CIFAR-10 reference dataset...")
if len(os.listdir("data/cifar10_real/FID")) < N_SAMPLES:
    transform = torchvision.transforms.Compose([torchvision.transforms.ToTensor()])
    real_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    for i, (img, _) in enumerate(real_set):
        if i >= N_SAMPLES: break
        torchvision.utils.save_image(img, f'data/cifar10_real/FID/{i:05d}.png')
    print(f"Extracted {N_SAMPLES} reference images.")
else:
    print(f"Found reference images. Skipping extraction.")

# =====================================================================
# 5. MATHEMATICALLY RIGOROUS DSG GENERATION LOOP
# =====================================================================
def dsg_sample_batch_optimized(pipe, D, batch_size, gamma=0.05, timestep_cutoff=(0.2, 0.8)):
    unet = pipe.unet
    scheduler = pipe.scheduler
    timesteps = scheduler.timesteps
    T = len(timesteps)

    # Initialize latents
    x_t = torch.randn(batch_size, 3, 32, 32).to(device)

    for i, t in enumerate(timesteps):
        t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)

        # 1. Compute UNet noise prediction
        with torch.no_grad():
            noise_pred = unet(x_t, t_batch).sample

        # 2. Step through the DDIM scheduler to get standard x_{t-1}
        with torch.no_grad():
            step_out = scheduler.step(noise_pred, t, x_t)
            x_prev = step_out.prev_sample

        # 3. Apply Manifold Guidance conditionally
        progress = i / T
        if timestep_cutoff[0] < progress < timestep_cutoff[1]:
            # Enable gradients on x_t to track the exact path to x0_hat
            x_t_grad = x_t.detach().requires_grad_(True)

            # Re-evaluate clean image prediction as a function of x_t_grad
            alpha_bar = scheduler.alphas_cumprod[t].to(device)
            x0_hat = (x_t_grad - (1 - alpha_bar).sqrt() * noise_pred) / alpha_bar.sqrt()
            x0_hat = x0_hat.clamp(-1, 1)

            # Compute log-odds score from the discriminator
            d_out = D(x0_hat).clamp(DELTA, 1 - DELTA)
            log_odds = (torch.log(d_out) - torch.log(1 - d_out)).sum()

            # Backpropwards directly to x_t
            D.zero_grad()
            log_odds.backward()
            grad_xt = x_t_grad.grad.detach()

            # Normalize the gradient vector to preserve stable scheduling dynamics
            grad_flat = grad_xt.view(batch_size, -1)
            grad_norm = grad_flat.norm(dim=1, keepdim=True).clamp(min=1e-8)
            grad_unit = (grad_flat / grad_norm).view_as(grad_xt)

            # Shift x_prev along the gradient direction scaled by current latent magnitude
            x_norm = x_prev.view(batch_size, -1).norm(dim=1).view(batch_size, 1, 1, 1)
            x_prev = x_prev + gamma * x_norm * grad_unit

        # Enforce boundary constraints and update latent
        x_t = x_prev.clamp(-1, 1)

    # Map from [-1, 1] back to PIL format
    imgs = ((x_t.cpu() + 1) / 2).clamp(0, 1)
    return [torchvision.transforms.ToPILImage()(imgs[j]) for j in range(batch_size)]

# =====================================================================
# 6. PIPELINE RUNS
# =====================================================================
print(f"\n[4/5] Generating {N_SAMPLES} Unguided DDIM baseline samples...")
pipe.scheduler.set_timesteps(100)
n_done = 0
pbar_unguided = tqdm(total=N_SAMPLES, desc="Unguided Baseline")
while n_done < N_SAMPLES:
    b = min(BATCH, N_SAMPLES - n_done)
    with torch.no_grad():
        out = pipe(batch_size=b, num_inference_steps=100, output_type="pil")
    for j, img in enumerate(out.images):
        img.save(f'samples/unguided_ddim/FID/{n_done+j:05d}.png')
    n_done += b
    pbar_unguided.update(b)
pbar_unguided.close()

print(f"\n[5/5] Generating {N_SAMPLES} Optimized DSG-guided samples...")
n_done = 0
pbar_guided = tqdm(total=N_SAMPLES, desc="DSG Guided")
while n_done < N_SAMPLES:
    b = min(BATCH, N_SAMPLES - n_done)
    imgs = dsg_sample_batch_optimized(pipe, D, b, gamma=GAMMA_DSG, timestep_cutoff=CUTOFF)
    for j, img in enumerate(imgs):
        img.save(f'samples/dsg_optimized/FID/{n_done+j:05d}.png')
    n_done += b
    pbar_guided.update(b)
pbar_guided.close()

# =====================================================================
# 7. METRIC EVALUATION
# =====================================================================
print("\nComputing absolute FID scores using clean-fid...")
fid_unguided = fid.compute_fid('samples/unguided_ddim/FID', 'data/cifar10_real/FID')
fid_dsg = fid.compute_fid('samples/dsg_optimized/FID', 'data/cifar10_real/FID')

print("\n" + "="*55)
print("               FINAL QUALITY VERIFICATION               ")
print("="*55)
print(f"  Baseline Unguided DDIM (100 steps) : FID = {fid_unguided:.2f}")
print(f"  Optimized DSG-Guided   (100 steps) : FID = {fid_dsg:.2f}")
print("-"*55)
print(f"  Net FID Improvement                : {fid_unguided - fid_dsg:+.2f} points")
print("="*55)

Running execution pipeline on: CUDA
Clearing old cache: samples/unguided_ddim
Clearing old cache: samples/dsg_optimized

[1/5] Loading diffusion pipeline components...


Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


[2/5] Loading weights from local file: dcgan_discriminator_cifar10.pt
Discriminator successfully loaded and set to evaluation mode.

[3/5] Checking real CIFAR-10 reference dataset...
Found reference images. Skipping extraction.

[4/5] Generating 1000 Unguided DDIM baseline samples...


Unguided Baseline:   0%|          | 0/1000 [00:00<?, ?it/s]

...

compute FID between two folders
Found 1000 images in the folder samples/dsg_optimized/FID


FID FID : 100%|██████████| 32/32 [00:12<00:00,  2.50it/s]


Found 1000 images in the folder data/cifar10_real/FID


FID FID : 100%|██████████| 32/32 [00:11<00:00,  2.70it/s]



               FINAL QUALITY VERIFICATION               
  Baseline Unguided DDIM (100 steps) : FID = 59.37
  Optimized DSG-Guided   (100 steps) : FID = 200.04
-------------------------------------------------------
  Net FID Improvement                : -140.68 points


In [ ]:
import os
import shutil
import torch
import torch.nn as nn
import numpy as np
import torchvision
from diffusers import DDPMPipeline, DDIMScheduler
from cleanfid import fid
from tqdm import tqdm

# =====================================================================
# 1. HARDWARE INITIALIZATION & CACHE PURGING
# =====================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running execution pipeline on: {device.upper()}")

# Clear old folders completely to prevent clean-fid cache pollution
dirs_to_clean = ["samples/unguided_ddim", "samples/dsg_optimized"]
for folder in dirs_to_clean:
    if os.path.exists(folder):
        print(f"Clearing old cache: {folder}")
        shutil.rmtree(folder)
    os.makedirs(f"{folder}/FID", exist_ok=True)

os.makedirs("data/cifar10_real/FID", exist_ok=True)

# =====================================================================
# 2. HYPERPARAMETER CONFIGURATION
# =====================================================================
N_SAMPLES = 1000       # Run 1000 for verification; 10000 for publication metrics
BATCH = 64
DELTA = 0.1
GAMMA_DSG = 0.05       # Stable step scale for normalized DDIM space guidance
CUTOFF = (0.2, 0.8)    # Active guidance window (20% to 80% progress)
LOCAL_WEIGHTS = 'dcgan_discriminator_cifar10.pt'

# =====================================================================
# 3. COMPONENT LOADING & SCHEDULER SWAP
# =====================================================================
print("\n[1/5] Loading diffusion pipeline components...")
pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32").to(device)

# THE CRITICAL STEP SWAP: Allow arbitrary 100-step trajectories mathematically
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.scheduler.set_timesteps(100)

print(f"[2/5] Loading weights from local file: {LOCAL_WEIGHTS}")
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, kernel_size=2, stride=1, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

D = Discriminator().to(device)
if os.path.exists(LOCAL_WEIGHTS):
    D.load_state_dict(torch.load(LOCAL_WEIGHTS, map_location=device))
    D.eval()
    print("Discriminator successfully loaded and set to evaluation mode.")
else:
    raise FileNotFoundError(f"Missing {LOCAL_WEIGHTS}! Place it in the current execution folder.")

# =====================================================================
# 4. REFERENCE DATASET VALIDATION
# =====================================================================
print("\n[3/5] Checking real CIFAR-10 reference dataset...")
if len(os.listdir("data/cifar10_real/FID")) < N_SAMPLES:
    transform = torchvision.transforms.Compose([torchvision.transforms.ToTensor()])
    real_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    for i, (img, _) in enumerate(real_set):
        if i >= N_SAMPLES: break
        torchvision.utils.save_image(img, f'data/cifar10_real/FID/{i:05d}.png')
    print(f"Extracted {N_SAMPLES} reference images.")
else:
    print(f"Found reference images. Skipping extraction.")

# =====================================================================
# 5. MATHEMATICALLY SOUND DSG SAMPLING (MANIFOLD TRACKING)
# =====================================================================
def dsg_sample_batch_optimized(pipe, D, batch_size, gamma=0.05, timestep_cutoff=(0.2, 0.8)):
    unet = pipe.unet
    scheduler = pipe.scheduler
    timesteps = scheduler.timesteps
    T = len(timesteps)

    # Standard Gaussian Initialization
    x_t = torch.randn(batch_size, 3, 32, 32).to(device)

    for i, t in enumerate(timesteps):
        t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)

        progress = i / T
        # Check if we are inside the active guidance window
        if timestep_cutoff[0] < progress < timestep_cutoff[1]:
            # THE CORE INTEGRATION FIX: Track the gradient across the UNet's manifold response
            x_t = x_t.detach().requires_grad_(True)
            noise_pred = unet(x_t, t_batch).sample

            alpha_bar = scheduler.alphas_cumprod[t].to(device)
            x0_hat = (x_t - (1 - alpha_bar).sqrt() * noise_pred) / alpha_bar.sqrt()
            x0_hat = x0_hat.clamp(-1, 1)

            # Formulate the Discriminator log-odds score objective
            d_out = D(x0_hat).clamp(DELTA, 1 - DELTA)
            log_odds = (torch.log(d_out) - torch.log(1 - d_out)).sum()

            # Backpropwards directly through the entire step graph to x_t
            grad_xt = torch.autograd.grad(log_odds, x_t)[0].detach()

            # Reshape and impose unit-norm boundary constraints to ensure scale stability
            grad_flat = grad_xt.view(batch_size, -1)
            grad_norm = grad_flat.norm(dim=1, keepdim=True).clamp(min=1e-8)
            grad_unit = (grad_flat / grad_norm).view_as(grad_xt)

            # Cleanly break graph links to free up GPU cache allocations
            x_t = x_t.detach()
            noise_pred = noise_pred.detach()

            # Step along standard deterministic DDIM trajectory
            step_out = scheduler.step(noise_pred, t, x_t)
            x_prev = step_out.prev_sample

            # Apply manifold displacement correction scaled relative to current latent magnitude
            x_norm = x_prev.view(batch_size, -1).norm(dim=1).view(batch_size, 1, 1, 1)
            x_prev = x_prev + gamma * x_norm * grad_unit
        else:
            # Outside guidance window: Fast unguided execution
            with torch.no_grad():
                noise_pred = unet(x_t, t_batch).sample
                step_out = scheduler.step(noise_pred, t, x_t)
                x_prev = step_out.prev_sample

        # Clamp pixels and carry over to the next loop step
        x_t = x_prev.clamp(-1, 1)

    # Convert tensors back to standard PIL structures
    imgs = ((x_t.cpu() + 1) / 2).clamp(0, 1)
    return [torchvision.transforms.ToPILImage()(imgs[j]) for j in range(batch_size)]

# =====================================================================
# 6. PIPELINE PROCESSING LOOPS
# =====================================================================
print(f"\n[4/5] Generating {N_SAMPLES} Unguided DDIM baseline samples...")
pipe.scheduler.set_timesteps(100)
n_done = 0
pbar_unguided = tqdm(total=N_SAMPLES, desc="Unguided Baseline")
while n_done < N_SAMPLES:
    b = min(BATCH, N_SAMPLES - n_done)
    with torch.no_grad():
        out = pipe(batch_size=b, num_inference_steps=100, output_type="pil")
    for j, img in enumerate(out.images):
        img.save(f'samples/unguided_ddim/FID/{n_done+j:05d}.png')
    n_done += b
    pbar_unguided.update(b)
pbar_unguided.close()

print(f"\n[5/5] Generating {N_SAMPLES} Optimized DSG-guided samples...")
n_done = 0
pbar_guided = tqdm(total=N_SAMPLES, desc="DSG Guided")
while n_done < N_SAMPLES:
    b = min(BATCH, N_SAMPLES - n_done)
    imgs = dsg_sample_batch_optimized(pipe, D, b, gamma=GAMMA_DSG, timestep_cutoff=CUTOFF)
    for j, img in enumerate(imgs):
        img.save(f'samples/dsg_optimized/FID/{n_done+j:05d}.png')
    n_done += b
    pbar_guided.update(b)
pbar_guided.close()

# =====================================================================
# 7. PERFORMANCE EVALUATION (CLEAN-FID)
# =====================================================================
print("\nComputing absolute FID scores using clean-fid...")
fid_unguided = fid.compute_fid('samples/unguided_ddim/FID', 'data/cifar10_real/FID')
fid_dsg = fid.compute_fid('samples/dsg_optimized/FID', 'data/cifar10_real/FID')

print("\n" + "="*55)
print("               FINAL QUALITY VERIFICATION               ")
print("="*55)
print(f"  Baseline Unguided DDIM (100 steps) : FID = {fid_unguided:.2f}")
print(f"  Optimized DSG-Guided   (100 steps) : FID = {fid_dsg:.2f}")
print("-"*55)
print(f"  Net FID Improvement                : {fid_unguided - fid_dsg:+.2f} points")
print("="*55)

Running execution pipeline on: CUDA
Clearing old cache: samples/unguided_ddim
Clearing old cache: samples/dsg_optimized

[1/5] Loading diffusion pipeline components...


Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


[2/5] Loading weights from local file: dcgan_discriminator_cifar10.pt
Discriminator successfully loaded and set to evaluation mode.

[3/5] Checking real CIFAR-10 reference dataset...
Found reference images. Skipping extraction.

[4/5] Generating 1000 Unguided DDIM baseline samples...


Unguided Baseline:   0%|          | 0/1000 [00:00<?, ?it/s]

...

compute FID between two folders
Found 1000 images in the folder samples/dsg_optimized/FID


FID FID : 100%|██████████| 32/32 [00:12<00:00,  2.53it/s]


Found 1000 images in the folder data/cifar10_real/FID


FID FID : 100%|██████████| 32/32 [00:11<00:00,  2.73it/s]



               FINAL QUALITY VERIFICATION               
  Baseline Unguided DDIM (100 steps) : FID = 60.04
  Optimized DSG-Guided   (100 steps) : FID = 197.42
-------------------------------------------------------
  Net FID Improvement                : -137.38 points


In [ ]:
import os
import shutil
import torch
import torch.nn as nn
import numpy as np
import torchvision
from diffusers import DDPMPipeline, DDIMScheduler
from cleanfid import fid
from tqdm import tqdm

# =====================================================================
# 1. HARDWARE & DIRECTORY PURGE
# =====================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running execution pipeline on: {device.upper()}")

dirs_to_clean = ["samples/unguided_ddim", "samples/dsg_optimized"]
for folder in dirs_to_clean:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(f"{folder}/FID", exist_ok=True)
os.makedirs("data/cifar10_real/FID", exist_ok=True)

# =====================================================================
# 2. THEORETICALLY SOUND HYPERPARAMETERS
# =====================================================================
N_SAMPLES = 1000
BATCH = 64
DELTA = 0.05           # Discriminator clipping bound (Assumption A3)

# Guidance Scale mapped to clean data space
GAMMA_DSG = 0.1
# Guide between 20% and 80% progress where the UNet structure is malleable
CUTOFF = (0.20, 0.80)
LOCAL_WEIGHTS = 'dcgan_discriminator_cifar10.pt'

# =====================================================================
# 3. COMPONENT MODEL INITIALIZATION
# =====================================================================
print("\n[1/5] Loading diffusion pipeline components...")
pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32").to(device)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.scheduler.set_timesteps(100)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, kernel_size=2, stride=1, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

D = Discriminator().to(device)
D.load_state_dict(torch.load(LOCAL_WEIGHTS, map_location=device))
D.eval()

if len(os.listdir("data/cifar10_real/FID")) < N_SAMPLES:
    transform = torchvision.transforms.Compose([torchvision.transforms.ToTensor()])
    real_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    for i, (img, _) in enumerate(real_set):
        if i >= N_SAMPLES: break
        torchvision.utils.save_image(img, f'data/cifar10_real/FID/{i:05d}.png')

# =====================================================================
# 4. MANIFOLD-CONSTRAINED TWEEDIE DSG SAMPLER
# =====================================================================
def dsg_sample_batch_tweedie(pipe, D, batch_size, gamma=0.1, timestep_cutoff=(0.20, 0.80)):
    unet = pipe.unet
    scheduler = pipe.scheduler
    timesteps = scheduler.timesteps
    T = len(timesteps)

    x_t = torch.randn(batch_size, 3, 32, 32).to(device)

    for i, t in enumerate(timesteps):
        t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)
        progress = i / T

        # 1. Run standard UNet score prediction
        with torch.no_grad():
            noise_pred = unet(x_t, t_batch).sample

        alpha_bar = scheduler.alphas_cumprod[t].to(device)

        # 2. Check if we are within the guidance window
        if timestep_cutoff[0] < progress < timestep_cutoff[1]:
            # Calculate the clean projected image x0_hat
            x0_hat = (x_t - (1 - alpha_bar).sqrt() * noise_pred) / alpha_bar.sqrt()
            x0_hat = x0_hat.clamp(-1, 1).detach().requires_grad_(True)

            # Evaluate log-odds on clean data space (matching your proof assumptions)
            d_out = D(x0_hat).clamp(DELTA, 1 - DELTA)
            log_odds = (torch.log(d_out) - torch.log(1 - d_out)).sum()

            # Compute gradient explicitly with respect to clean x0 space
            grad_x0 = torch.autograd.grad(log_odds, x0_hat)[0].detach()

            # Stabilize and normalize the guidance vector
            grad_flat = grad_x0.view(batch_size, -1)
            grad_norm = grad_flat.norm(dim=1).view(batch_size, 1, 1, 1).clamp(min=1e-5)
            grad_unit = grad_x0 / grad_norm

            # Apply guidance directly to the clean image estimate
            x0_hat_guided = x0_hat.detach() + gamma * grad_unit
            x0_hat_guided = x0_hat_guided.clamp(-1, 1)

            # 3. Tweedie's Formula: Reconstruct the corrected noise prediction vector
            # This links your clean space math back to the diffusion latent space smoothly
            noise_pred_corrected = (x_t - alpha_bar.sqrt() * x0_hat_guided) / (1 - alpha_bar).sqrt()
            noise_pred_final = noise_pred_corrected.detach()
        else:
            noise_pred_final = noise_pred

        # 4. Standard DDIM step transition using the corrected noise vector
        with torch.no_grad():
            step_out = scheduler.step(noise_pred_final, t, x_t)
            x_t = step_out.prev_sample.clamp(-1, 1)

    imgs = ((x_t.cpu() + 1) / 2).clamp(0, 1)
    return [torchvision.transforms.ToPILImage()(imgs[j]) for j in range(batch_size)]

# =====================================================================
# 5. PIPELINE RUN LOOPS
# =====================================================================
print(f"\n[4/5] Generating {N_SAMPLES} Unguided DDIM baseline samples...")
pipe.scheduler.set_timesteps(100)
n_done = 0
pbar_unguided = tqdm(total=N_SAMPLES, desc="Unguided Baseline")
while n_done < N_SAMPLES:
    b = min(BATCH, N_SAMPLES - n_done)
    with torch.no_grad():
        out = pipe(batch_size=b, num_inference_steps=100, output_type="pil")
    for j, img in enumerate(out.images):
        img.save(f'samples/unguided_ddim/FID/{n_done+j:05d}.png')
    n_done += b
    pbar_unguided.update(b)
pbar_unguided.close()

print(f"\n[5/5] Generating {N_SAMPLES} Manifold-Constrained DSG samples...")
n_done = 0
pbar_guided = tqdm(total=N_SAMPLES, desc="Tweedie DSG Loop")
while n_done < N_SAMPLES:
    b = min(BATCH, N_SAMPLES - n_done)
    imgs = dsg_sample_batch_tweedie(pipe, D, b, gamma=GAMMA_DSG, timestep_cutoff=CUTOFF)
    for j, img in enumerate(imgs):
        img.save(f'samples/dsg_optimized/FID/{n_done+j:05d}.png')
    n_done += b
    pbar_guided.update(b)
pbar_guided.close()

# =====================================================================
# 6. QUALITY METRICS EVALUATION
# =====================================================================
print("\nComputing absolute FID scores using clean-fid...")
fid_unguided = fid.compute_fid('samples/unguided_ddim/FID', 'data/cifar10_real/FID')
fid_dsg = fid.compute_fid('samples/dsg_optimized/FID', 'data/cifar10_real/FID')

print("\n" + "="*55)
print(f"  Baseline Unguided DDIM : FID = {fid_unguided:.2f}")
print(f"  Tweedie DSG-Guided     : FID = {fid_dsg:.2f}")
print(f"  Net FID Improvement    : {fid_unguided - fid_dsg:+.2f} points")
print("="*55)

Running execution pipeline on: CUDA

[1/5] Loading diffusion pipeline components...


Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.



[4/5] Generating 1000 Unguided DDIM baseline samples...


Unguided Baseline:   0%|          | 0/1000 [00:00<?, ?it/s]

...

compute FID between two folders
Found 1000 images in the folder samples/dsg_optimized/FID


FID FID : 100%|██████████| 32/32 [00:11<00:00,  2.70it/s]


Found 1000 images in the folder data/cifar10_real/FID


FID FID : 100%|██████████| 32/32 [00:12<00:00,  2.60it/s]



  Baseline Unguided DDIM : FID = 59.37
  Tweedie DSG-Guided     : FID = 198.57
  Net FID Improvement    : -139.20 points


In [ ]:
import os
import shutil
import torch
import torch.nn as nn
import numpy as np
import torchvision
from diffusers import DDPMPipeline, DDIMScheduler
from cleanfid import fid
from tqdm import tqdm

# =====================================================================
# 1. HARDWARE & WORKSPACE
# =====================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running Option A execution on: {device.upper()}")

# Clean output directories
for folder in ["samples/unguided_ddim", "samples/dsg_time_latent"]:
    if os.path.exists(folder): shutil.rmtree(folder)
    os.makedirs(f"{folder}/FID", exist_ok=True)
os.makedirs("data/cifar10_real/FID", exist_ok=True)

# =====================================================================
# 2. TIME-CONDITIONED DISCRIMINATOR ARCHITECTURE
# =====================================================================
class TimeConditionedDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.init_conv = nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1)
        # Embedding layer to inject timestep t into the discriminator's feature space
        self.time_mlp = nn.Sequential(nn.Linear(1, 64), nn.SiLU(), nn.Linear(64, 64))

        self.main_net = nn.Sequential(
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x, t):
        t_embed = self.time_mlp(t.float().view(-1, 1)).view(-1, 64, 1, 1)
        h = self.init_conv(x) + t_embed
        return self.main_net(h)

# =====================================================================
# 3. SAMPLING FUNCTION (Option A: Latent Guidance)
# =====================================================================
def dsg_sample_time_latent(pipe, D, batch_size, gamma=0.005, cutoff=(0.25, 0.80)):
    scheduler = pipe.scheduler
    x_t = torch.randn(batch_size, 3, 32, 32).to(device)

    for i, t in enumerate(scheduler.timesteps):
        t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)
        progress = i / len(scheduler.timesteps)

        # Guided phase: Evaluate gradient directly on latent x_t at timestep t
        if cutoff[0] < progress < cutoff[1]:
            x_t = x_t.detach().requires_grad_(True)

            # Log-odds guidance term ∇_x log(D(x,t) / (1 - D(x,t)))
            d_out = D(x_t, t_batch).clamp(0.05, 0.95)
            log_odds = (torch.log(d_out) - torch.log(1 - d_out)).sum()

            grad_xt = torch.autograd.grad(log_odds, x_t)[0].detach()
            x_t = x_t.detach()

            # Apply guidance to the latent trajectory
            grad_norm = grad_xt.view(batch_size, -1).norm(dim=1).view(batch_size, 1, 1, 1).clamp(min=1e-5)

            with torch.no_grad():
                noise_pred = pipe.unet(x_t, t_batch).sample
                x_prev = scheduler.step(noise_pred, t, x_t).prev_sample
                # Step displacement using conditioned gradient
                x_t = (x_prev + gamma * (grad_xt / grad_norm)).clamp(-1, 1)
        else:
            with torch.no_grad():
                noise_pred = pipe.unet(x_t, t_batch).sample
                x_t = scheduler.step(noise_pred, t, x_t).prev_sample.clamp(-1, 1)

    return [torchvision.transforms.ToPILImage()(((x_t[j].cpu() + 1) / 2).clamp(0, 1)) for j in range(batch_size)]

# =====================================================================
# 4. EXECUTION PIPELINE
# =====================================================================
pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32").to(device)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.scheduler.set_timesteps(100)
D = TimeConditionedDiscriminator().to(device)

print(f"\nGenerating 1000 samples using Option A (Time-Conditioned Latent Guidance)...")
# Note: Ensure you have trained weights for your TimeConditionedDiscriminator here.
# For demonstration, we assume D is initialized.
n_done = 0
while n_done < 1000:
    b = min(64, 1000 - n_done)
    imgs = dsg_sample_time_latent(pipe, D, b)
    for j, img in enumerate(imgs):
        img.save(f'samples/dsg_time_latent/FID/{n_done+j:05d}.png')
    n_done += b

Running Option A execution on: CUDA


Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.



Generating 1000 samples using Option A (Time-Conditioned Latent Guidance)...


In [ ]:
import os
import shutil
import torch
import torch.nn as nn
import numpy as np
import torchvision
from diffusers import DDPMPipeline, DDIMScheduler
from cleanfid import fid
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset

# =====================================================================
# 1. SETUP & WORKSPACE
# =====================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running complete pipeline on: {device.upper()}")

# Reset directories
for folder in ["samples/unguided_ddim", "samples/dsg_time_latent"]:
    if os.path.exists(folder): shutil.rmtree(folder)
    os.makedirs(f"{folder}/FID", exist_ok=True)
os.makedirs("data/cifar10_real/FID", exist_ok=True)

# =====================================================================
# 2. TIME-CONDITIONED DISCRIMINATOR
# =====================================================================
class TimeConditionedDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.init_conv = nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1)
        self.time_mlp = nn.Sequential(nn.Linear(1, 64), nn.SiLU(), nn.Linear(64, 64))
        self.main_net = nn.Sequential(
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x, t):
        t_embed = self.time_mlp(t.float().view(-1, 1)).view(-1, 64, 1, 1)
        h = self.init_conv(x) + t_embed
        return self.main_net(h)

# =====================================================================
# 3. TRAINING & SAMPLING LOGIC
# =====================================================================
def train_discriminator(D, pipe, real_loader, epochs=3):
    D.train()
    optimizer = torch.optim.Adam(D.parameters(), lr=1e-4)
    criterion = nn.BCELoss()
    for epoch in range(epochs):
        for real_imgs, _ in real_loader:
            optimizer.zero_grad()
            t = torch.randint(0, 100, (real_imgs.size(0),), device=device)
            noise = torch.randn_like(real_imgs.to(device))
            x_t = pipe.scheduler.add_noise(real_imgs.to(device), noise, t)

            loss = criterion(D(real_imgs.to(device), torch.zeros_like(t)), torch.ones(real_imgs.size(0), 1, 1, 1, device=device)) + \
                   criterion(D(x_t, t), torch.zeros(real_imgs.size(0), 1, 1, 1, device=device))
            loss.backward(); optimizer.step()
    D.eval()

def dsg_sample(pipe, D, batch_size, gamma=0.005):
    scheduler = pipe.scheduler
    x_t = torch.randn(batch_size, 3, 32, 32).to(device)
    for i, t in enumerate(scheduler.timesteps):
        x_t = x_t.detach().requires_grad_(True)
        t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)

        # Guided step based on log-odds gradient
        d_out = D(x_t, t_batch).clamp(0.05, 0.95)
        log_odds = (torch.log(d_out) - torch.log(1 - d_out)).sum()
        grad = torch.autograd.grad(log_odds, x_t)[0].detach()
        x_t = x_t.detach()

        noise_pred = pipe.unet(x_t, t_batch).sample
        x_prev = scheduler.step(noise_pred, t, x_t).prev_sample
        x_t = (x_prev + gamma * (grad / grad.norm(dim=(1,2,3), keepdim=True).clamp(min=1e-5))).clamp(-1, 1)
    return [torchvision.transforms.ToPILImage()(((x_t[j].cpu() + 1) / 2).clamp(0, 1)) for j in range(batch_size)]

# =====================================================================
# 4. EXECUTION
# =====================================================================
pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32").to(device)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
D = TimeConditionedDiscriminator().to(device)
real_loader = DataLoader(torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=torchvision.transforms.ToTensor()), batch_size=64, shuffle=True)

train_discriminator(D, pipe, real_loader)

# Generate Baseline
for i in tqdm(range(0, 1000, 64)):
    out = pipe(batch_size=min(64, 1000-i), output_type="pil").images
    for j, img in enumerate(out): img.save(f'samples/unguided_ddim/FID/{i+j:05d}.png')

# Generate Guided
for i in tqdm(range(0, 1000, 64)):
    imgs = dsg_sample(pipe, D, min(64, 1000-i))
    for j, img in enumerate(imgs): img.save(f'samples/dsg_time_latent/FID/{i+j:05d}.png')

print(f"FID Score: {fid.compute_fid('samples/dsg_time_latent/FID', 'data/cifar10_real/FID'):.2f}")

Running complete pipeline on: CUDA


Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  6%|▋         | 1/16 [02:51<42:45, 171.03s/it]

...

  0%|          | 0/1000 [00:00<?, ?it/s]

 94%|█████████▍| 15/16 [42:41<02:50, 170.05s/it]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]


RuntimeError: linalg.matrix_norm: dim must be a 2-tuple. Got 1 2 3

In [ ]:
import os
import shutil
import torch
import torch.nn as nn
import numpy as np
import torchvision
from diffusers import DDPMPipeline, DDIMScheduler
from cleanfid import fid
from tqdm import tqdm
from torch.utils.data import DataLoader
import gc # Import garbage collector

# =====================================================================
# 1. TIME-CONDITIONED DISCRIMINATOR (The "Option A" Architecture)
# =====================================================================
class TimeConditionedDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.init_conv = nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1)
        self.time_mlp = nn.Sequential(nn.Linear(1, 64), nn.SiLU(), nn.Linear(64, 64))
        self.main_net = nn.Sequential(
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x, t):
        # Revert: Remove explicit float16 cast for time_mlp input to avoid CUDA error
        t_embed = self.time_mlp(t.float().view(-1, 1)).view(-1, 64, 1, 1)
        h = self.init_conv(x) + t_embed
        return self.main_net(h)

# =====================================================================
# 2. STABILIZED GUIDED SAMPLER
# =====================================================================
def dsg_sample_stabilized(pipe, D, batch_size, gamma=0.005, skip=5):
    scheduler = pipe.scheduler
    # Revert: Initialize x_t as float32
    x_t = torch.randn(batch_size, 3, 32, 32).to('cuda')
    cached_grad = torch.zeros_like(x_t)

    for i, t in enumerate(scheduler.timesteps):
        t_batch = torch.full((batch_size,), t, device='cuda', dtype=torch.long)

        # 1. Update guidance gradient every 'skip' steps
        if i % skip == 0:
            x_t = x_t.detach().requires_grad_(True)
            d_out = D(x_t, t_batch).clamp(0.05, 0.95)
            log_odds = (torch.log(d_out) - torch.log(1 - d_out)).sum()
            grad = torch.autograd.grad(log_odds, x_t)[0].detach()

            # Normalize gradient to ensure stable scale relative to DDPM score
            grad_norm = torch.linalg.vector_norm(grad, dim=(1, 2, 3), keepdim=True).clamp(min=1e-5)
            cached_grad = grad / grad_norm
            x_t = x_t.detach()

        # 2. Apply guidance with linear decay (strongest at start, weakest at end)
        decay = (1.0 - (i / len(scheduler.timesteps)))
        noise_pred = pipe.unet(x_t, t_batch).sample
        x_prev = scheduler.step(noise_pred, t, x_t).prev_sample
        x_t = (x_prev + (gamma * decay) * cached_grad).clamp(-1, 1)

    return [torchvision.transforms.ToPILImage()(((x_t[j].cpu() + 1) / 2).clamp(0, 1)) for j in range(batch_size)]

# =====================================================================
# 3. FULL EXECUTION CONTROLLER
# =====================================================================
device = 'cuda'
torch.cuda.empty_cache() # Clear GPU cache at the very beginning
gc.collect() # Explicitly call garbage collector
torch.cuda.empty_cache() # Clear again after gc.collect()

# Revert: Load DDPMPipeline as float32. Explicitly set torch_dtype=torch.float32 for strictness.
# Changed: Load on CPU first, then move to device, explicitly with device_map='cpu'
pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32", torch_dtype=torch.float32, device_map='cpu')
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.scheduler.set_timesteps(50) # Reduced steps for faster convergence
pipe = pipe.to(device) # Move to device AFTER initial setup
D = TimeConditionedDiscriminator().to(device)
# Revert: Remove conversion of discriminator to float16

# A. Train Discriminator
real_loader = DataLoader(torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=torchvision.transforms.ToTensor()), batch_size=64, shuffle=True)
D.train()
optimizer = torch.optim.Adam(D.parameters(), lr=1e-4)
for real_imgs, _ in tqdm(real_loader, desc="Training Discriminator"):
    optimizer.zero_grad()
    # Revert: Ensure real_imgs are float32 and target tensors for BCELoss are float32
    real_imgs = real_imgs.to(device)
    t = torch.randint(0, 50, (real_imgs.size(0),), device=device)
    noise = torch.randn_like(real_imgs) # noise will be float32
    x_t = pipe.scheduler.add_noise(real_imgs, noise, t)

    loss = nn.BCELoss()(D(real_imgs, torch.zeros_like(t)), torch.ones(real_imgs.size(0), 1, 1, 1, device=device)) + \
           nn.BCELoss()(D(x_t, t), torch.zeros(real_imgs.size(0), 1, 1, 1, device=device))
    loss.backward(); optimizer.step()
D.eval()

# B. Generate Samples
os.makedirs('samples/dsg_final/FID', exist_ok=True)
torch.cuda.empty_cache() # Clear GPU cache before generation
for i in tqdm(range(0, 500, 8), desc="Generating Guided Samples"): # Reduced batch size to 8
    imgs = dsg_sample_stabilized(pipe, D, min(8, 500-i)) # Reduced batch size to 8
    for j, img in enumerate(imgs): img.save(f'samples/dsg_final/FID/{i+j:05d}.png')

print(f"Final FID: {fid.compute_fid('samples/dsg_final/FID', 'data/cifar10_real/FID'):.2f}")

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import os
import torch
import torch.nn as nn
import torchvision
from diffusers import DDPMPipeline, DDIMScheduler
from cleanfid import fid
from tqdm import tqdm
from torch.utils.data import DataLoader
import gc # Import garbage collector

# Setup
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
device = 'cuda'

# 1. STABLE DISCRIMINATOR (Removed Sigmoid for BCEWithLogitsLoss)
class StableDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.Flatten(), nn.Linear(512 * 4 * 4, 1)
        )
    def forward(self, x): return self.net(x)

# 2. STABLE ENERGY SAMPLER
def dsg_sample_stable(pipe, D, batch_size, gamma=0.0001):
    scheduler = pipe.scheduler
    x_t = torch.randn(batch_size, 3, 32, 32).to(device)

    for i, t in enumerate(scheduler.timesteps):
        x_t = x_t.detach().requires_grad_(True)
        # Energy via BCEWithLogits (stable)
        logits = D(x_t)
        energy = nn.functional.binary_cross_entropy_with_logits(logits, torch.ones_like(logits))
        grad = torch.autograd.grad(energy, x_t, retain_graph=False)[0].detach()

        # Safe normalization
        grad = grad / (grad.norm(dim=(1,2,3), keepdim=True).clamp(min=1e-8))
        decay = (1.0 - (i / len(scheduler.timesteps)))

        noise_pred = pipe.unet(x_t, torch.full((batch_size,), t, device=device)).sample
        x_prev = scheduler.step(noise_pred, t, x_t).prev_sample
        x_t = (x_prev - (gamma * decay * grad)).clamp(-1, 1)

    return [torchvision.transforms.ToPILImage()(((x_t[j].cpu() + 1) / 2).clamp(0, 1)) for j in range(batch_size)]

# 3. EXECUTION
# Clear GPU cache and collect garbage before loading a potentially large model
torch.cuda.empty_cache()
gc.collect()

# Load the pipeline to CPU first, then explicitly move to device
# This aims to ensure memory is allocated and managed carefully
pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32", torch_dtype=torch.float32) # Load to CPU (default)
pipe.to(device) # Explicitly move to the desired device

pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.scheduler.set_timesteps(50)
D = StableDiscriminator().to(device)
optimizer = torch.optim.Adam(D.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

# Train
loader = DataLoader(torchvision.datasets.CIFAR10('./data', train=True, download=True,
                    transform=torchvision.transforms.ToTensor()), batch_size=32, shuffle=True)

D.train()
# The `total=100` argument in tqdm will limit the loop to 100 iterations.
# The original `if _ == 100: break` was a typo, `_` is the batch itself.
for real_imgs, _ in tqdm(loader, total=100, desc="Training"):
    optimizer.zero_grad()
    with torch.no_grad():
        # pipe(...).images returns a list of PIL images, convert to tensors and move to device
        fake_imgs_pil = pipe(batch_size=real_imgs.size(0)).images
        fake_imgs = torch.stack([torchvision.transforms.ToTensor()(img) for img in fake_imgs_pil]).to(device)

    # Ensure all tensors for criterion are float32
    loss = criterion(D(real_imgs.to(device)), torch.ones(real_imgs.size(0), 1, device=device)) + \
           criterion(D(fake_imgs), torch.zeros(real_imgs.size(0), 1, device=device))
    loss.backward(); optimizer.step()

# Clear GPU cache after training
torch.cuda.empty_cache()
gc.collect()

# Generate
os.makedirs('samples/dsg_stable/FID', exist_ok=True)
for i in tqdm(range(0, 200, 32)):
    imgs = dsg_sample_stable(pipe, D, min(32, 200-i))
    for j, img in enumerate(imgs): img.save(f'samples/dsg_stable/FID/{i+j:05d}.png')

print(f"Final FID: {fid.compute_fid('samples/dsg_stable/FID', 'data/cifar10_real/FID'):.2f}")

Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import os
import torch
import torch.nn as nn
import torchvision
from diffusers import DDPMPipeline, DDIMScheduler
from cleanfid import fid
from tqdm import tqdm
from torch.utils.data import DataLoader

device = 'cuda'

# 1. ROBUST DISCRIMINATOR (ResNet-Block Style for better manifold learning)
class DeepDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.Conv2d(512, 1, 4, 1, 0), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).view(-1, 1)

# 2. ENERGY-BASED GUIDANCE
def dsg_sample_energy(pipe, D, batch_size, gamma=0.0001):
    scheduler = pipe.scheduler
    x_t = torch.randn(batch_size, 3, 32, 32).to(device)

    for i, t in enumerate(scheduler.timesteps):
        x_t = x_t.detach().requires_grad_(True)
        logits = D(x_t)
        # Minimize Energy (BCE Loss) to align with data manifold
        energy = nn.functional.binary_cross_entropy(logits, torch.ones_like(logits))
        grad = torch.autograd.grad(energy, x_t)[0].detach()

        # Normalize and decay: Guidance should be very subtle
        # FIX: Use torch.linalg.vector_norm for multi-dimensional norm
        grad = grad / (torch.linalg.vector_norm(grad, dim=(1,2,3), keepdim=True).clamp(min=1e-5))
        decay = (1.0 - (i / len(scheduler.timesteps)))

        noise_pred = pipe.unet(x_t, torch.full((batch_size,), t, device=device)).sample
        x_prev = scheduler.step(noise_pred, t, x_t).prev_sample
        x_t = (x_prev - (gamma * decay * grad)).clamp(-1, 1)
    return [torchvision.transforms.ToPILImage()(((x_t[j].cpu() + 1) / 2).clamp(0, 1)) for j in range(batch_size)]

# 3. COMPLETE PIPELINE
# Revert: Load pipeline as float32. Explicitly set torch_dtype=torch.float32 for strictness.
# Changed: Load on CPU first, then move to device, explicitly with device_map='cpu'
pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32", torch_dtype=torch.float32, device_map='cpu')
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.scheduler.set_timesteps(50)
pipe = pipe.to(device) # Move to device AFTER initial setup
D = DeepDiscriminator().to(device)
# Revert: Remove conversion of discriminator to float16
optimizer = torch.optim.Adam(D.parameters(), lr=1e-4)

# Train on REAL vs GENERATED FAKES (Manifold Alignment)
real_loader = DataLoader(torchvision.datasets.CIFAR10(root='./data', train=True, download=True,
                         transform=torchvision.transforms.ToTensor()), batch_size=32, shuffle=True)

D.train()
print("Training discriminator to align with DDPM manifold...")
for real_imgs, _ in tqdm(real_loader, total=200): # Train for 200 batches
    optimizer.zero_grad()
    with torch.no_grad():
        # Revert: Cast real_imgs to float32
        real_imgs = real_imgs.to(device)
        fake_imgs = pipe(batch_size=real_imgs.size(0), output_type="tensor").images.to(device)

    # Revert: Ensure target tensors for BCELoss are also float32
    loss = nn.BCELoss()(D(real_imgs), torch.ones(real_imgs.size(0), 1, device=device)) + \
           nn.BCELoss()(D(fake_imgs), torch.zeros(real_imgs.size(0), 1, device=device))
    loss.backward(); optimizer.step()
    if real_imgs.shape[0] == 0: break # Use real_imgs for shape check

D.eval()
torch.cuda.empty_cache()

# Generate Guided Samples
os.makedirs('samples/dsg_final/FID', exist_ok=True)
for i in tqdm(range(0, 200, 32)): # Assuming 200 samples are desired
    imgs = dsg_sample_energy(pipe, D, min(32, 200-i))
    for j, img in enumerate(imgs): img.save(f'samples/dsg_final/FID/{i+j:05d}.png')

print(f"Final FID: {fid.compute_fid('samples/dsg_final/FID', 'data/cifar10_real/FID'):.2f}")

Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import os
import torch
import torch.nn as nn
import torchvision
from diffusers import DDPMPipeline, DDIMScheduler
from cleanfid import fid
from tqdm import tqdm
from torch.utils.data import DataLoader

# CRITICAL: Restart your runtime before running this.
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
device = 'cuda'

# 1. ROBUST ARCHITECTURE
class RobustDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 3, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 3, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 1, 4, 1, 0)
        )
    def forward(self, x): return self.net(x.clamp(-1, 1))

# 2. STABLE ENERGY SAMPLER
def dsg_sample_stable(pipe, D, batch_size, gamma=0.0001):
    scheduler = pipe.scheduler
    x_t = torch.randn(batch_size, 3, 32, 32).to(device)
    for i, t in enumerate(scheduler.timesteps):
        x_t = x_t.detach().requires_grad_(True)
        logits = D(x_t).mean()
        # Energy minimization: Softplus is safer than Logit-Loss
        energy = torch.nn.functional.softplus(-logits)
        grad = torch.autograd.grad(energy, x_t, retain_graph=False)[0].detach()
        grad = grad / (grad.view(batch_size, -1).norm(dim=1, keepdim=True).view(-1, 1, 1, 1).clamp(min=1e-6))

        noise_pred = pipe.unet(x_t, torch.full((batch_size,), t, device=device)).sample
        x_prev = scheduler.step(noise_pred, t, x_t).prev_sample
        x_t = (x_prev - (gamma * (1 - i/50) * grad)).detach().clamp(-1, 1)
    return [torchvision.transforms.ToPILImage()(((x_t[j].cpu() + 1) / 2).clamp(0, 1)) for j in range(batch_size)]

# 3. CLEAN EXECUTION
# Ensure you have restarted the kernel to clear the bad GPU context
pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32", use_safetensors=False).to(device)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.scheduler.set_timesteps(50)
D = RobustDiscriminator().to(device)
optimizer = torch.optim.Adam(D.parameters(), lr=1e-4)

# Training with numerical safety
loader = DataLoader(torchvision.datasets.CIFAR10('./data', train=True, download=True,
                    transform=torchvision.transforms.ToTensor()), batch_size=16, shuffle=True)

D.train()
for i, (real_imgs, _) in enumerate(tqdm(loader, desc="Training")):
    if i > 50: break
    optimizer.zero_grad()
    with torch.no_grad():
        fake_imgs = pipe(batch_size=16).images.to(device)
    # Energy-based loss: Real -> High Logit, Fake -> Low Logit
    loss = torch.nn.functional.softplus(-D(real_imgs.to(device))).mean() + \
           torch.nn.functional.softplus(D(fake_imgs)).mean()
    loss.backward()
    # Gradient clipping to prevent NaN
    torch.nn.utils.clip_grad_norm_(D.parameters(), 1.0)
    optimizer.step()

# Generate
os.makedirs('samples/dsg_stable/FID', exist_ok=True)
for i in tqdm(range(0, 128, 16)):
    imgs = dsg_sample_stable(pipe, D, 16)
    for j, img in enumerate(imgs): img.save(f'samples/dsg_stable/FID/{i+j:05d}.png')

print(f"Final FID: {fid.compute_fid('samples/dsg_stable/FID', 'data/cifar10_real/FID'):.2f}")

Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
